In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
#Read the dataset Q1_data.csv using read_csv()

food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)

In [ ]:
# Task 2: Write your code here:
#Inspect the first few rows using head()
df_food.head()

In [ ]:
# Task 3: Write your code here:
#Display dataset information using info()
df_food.info()

In [ ]:
# Task 4: Write your code here:
#Show statistical description using describe()
df_food.describe()

In [ ]:
# Task 5: Write your code here:
# Plot the target distribution (delivery_time)
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Price Distribution')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
#Drop the 'Order_ID' column from the data
aim = 'Order_ID'
df_food = df_food.drop(aim, axis=1)

In [ ]:
# Task 2: Write your code here:
#Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )

# Analyze missing values
missing_percentage = (df_food.isnull().sum() / len(df_food)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

# Since we have missing values we need to deal with them below-->

df_clean = df_food.copy()
print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=['Delivery_Time', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs'])
print(f"After dropping missing: {df_clean.shape}")


In [ ]:
# Task 3: Write your code here:
#Check and remove duplicates if any exist
print("Checking for duplicate rows...")
duplicate_rows = df_clean.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df_clean[col] = df_clean[col].fillna('unknown')

# Fill cylinders with mode - discrete feature, mode is most representative
df_clean['Delivery_Time'] = df_clean['Delivery_Time'].fillna(df_clean['Delivery_Time'].mode()[0])
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mode()[0])

print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:
# Task 4: Write your code here:
#Encode categorical variables if needed (Bonus if used One Hot Encoding)
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder
from sklearn.preprocessing import StandardScaler, LabelEncoder


# They are: Weather	Traffic_Level	Time_of_Day	Vehicle_Type

label_encoder = LabelEncoder() # Instantiate LabelEncoder
df_clean["Weather"] = label_encoder.fit_transform(df_clean["Weather"])
df_clean["Traffic_Level"] = label_encoder.fit_transform(df_clean["Traffic_Level"])
df_clean["Time_of_Day"] = label_encoder.fit_transform(df_clean["Time_of_Day"])
df_clean["Vehicle_Type"] = label_encoder.fit_transform(df_clean["Vehicle_Type"])


print('\nData after encoding:\n', df_clean["Weather"]) #show after encoding

In [ ]:
# Task 5: Write your code here:
#Apply feature scaling for all features (Use StandardScaler)
feature_cols = ['Distance_km',	'Weather',	'Traffic_Level',	'Time_of_Day',	'Vehicle_Type',	'Preparation_Time_min',	'Courier_Experience_yrs']
scaler = StandardScaler()
df_clean_scaled = scaler.fit_transform(df_clean[feature_cols])

print(f"\nScaled ranges - Min: {df_clean_scaled.min():.2f}, Max: {df_clean_scaled.max():.2f}")

In [ ]:
# Task 6: Write your code here:
#Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

# From above we have this that it is imbalnced:
# Plot the target distribution (delivery_time)
plt.figure(figsize=(10, 5))
plt.hist(df_clean['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Price Distribution')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
#1. Split the dataset into features (X) and target (y)

# Define features (X) and target (y)
X = pd.DataFrame(df_clean_scaled.copy())
y = pd.DataFrame(df_clean['Delivery_Time'].copy())

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import StratifiedKFold


models = {
"Random Forest Regressor": RandomForestRegressor(
n_estimators=200, # Number of trees in the forest
max_depth=10, # Maximum depth of trees
min_samples_split=2, # Minimum samples to split
min_samples_leaf=1, # Minimum samples at leaf
n_jobs=-1, # Use all CPU cores (-1 = all cores)
random_state=42 # Seed
)
}


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {name: {'mse': [], 'mae': [], 'rmse': [], 'r2': []} for name in models}

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
  X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
  y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

  for model_name, model in models.items():
    # Train
    model.fit(X_train, y_train)
    # Predict
    y_pred = model.predict(X_test)
    # Evaluate
    mae = mean_absolute_error(y_test, y_pred)
    # Store results
    results[model_name]['mae'].append(mae)


# Print average results
for model_name in results:
  print(f"\n{model_name}:")
  print(f" MAE: {np.mean(results[model_name]['mae']):.4f}")

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 8))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--', linewidth=2)
plt.xlabel("Actual y_test (Ground Truth)")
plt.ylabel("Predicted y_pred")
plt.title("Linear Regression: Predictions vs. Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: